In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

windows = True

if windows:
    workspace_path = "P:/workspaces/lg-ukbiobank/projects/"
else:
    workspace_path = "/data/workspaces/lag/workspaces/lg-ukbiobank/projects/"

In [2]:
def get_file_names_old(gcta_path):
    rois = ["lifg", "lstg", "rifg", "rstg"]
    #roi_rmtg_grad_1_gcta_6tsms_N23204.tsv_resid_norm_N23181.tsv_9.hsq
    var_names = ["roi_{0}_grad_{1}".format(roi, x) for roi in rois for x in range(2)]
    file_names = [os.path.join(gcta_path, x+"_gcta_*.hsq") for x in var_names]   
    return file_names, var_names

def get_file_names(fn):
    return sorted(glob.glob(fn.format("*")))
    
def get_var_names(fn):
    df = pd.read_csv(fn, sep=" |\t", nrows=2)
    return df.columns[2:]

def get_var_numbers(fn_list):
    return [int(os.path.splitext(x)[0].split("_")[-1]) for x in fn_list]

def get_files(fn, no_files):
    
    print("{0} files missing, {1} present".format(no_files - len(fns), len(fns)))
    return fns

def read_h2(data, fn, var_name):
    try:
        #load h2 data
        in_data = pd.read_csv(fn, sep="\t", float_precision='high')
        
        #add to dataframe
        data.loc[var_name, "Vg"] = in_data["Variance"][0]
        data.loc[var_name, "Vg_SE"] = in_data["SE"][0]
        data.loc[var_name, "Ve"] = in_data["Variance"][1]
        data.loc[var_name, "Ve_SE"] = in_data["SE"][1]
        data.loc[var_name, "logL"] = in_data["Variance"][4]
        data.loc[var_name, "P"] = in_data["Variance"][8]
        
    except FileNotFoundError:
        print("{} not found".format(fn))
            
    return data
    
def load_all_h2s(fn, var_names):
    

    data = pd.DataFrame(index=var_names, columns=["Vg", "Vg_SE", "Ve", "Ve_SE", "logL", "P"], dtype=np.float64)
    
    print("Reading h2")
    for i in range(len(var_names)):
        n=i+1
        data = read_h2(data, fn.format(n), var_names[i])

    return data

def to_list(in_list, fn):
    with open(fn, "w") as file:
        for row in in_list:
            file.write(str(row)+'\n')

In [3]:
h2_data = pd.read_csv(os.path.join(workspace_path, "CONGRADS_rest", "results", "heritability_df_final.tsv"), sep="\t", index_col=0)

In [7]:
h2_data["sig"] = h2_data["P"] < 0.05
h2_data[h2_data["idp_type"] == "roi_melodic_ic"].head(n=12)

,Vg,Vg_SE,Ve,Ve_SE,logL,P,atlas,idp_type,g,roi,sig
melodic_g1_cmap_lifg_oic_1,0.093554,0.008950,0.905696,0.010314,-22638.890,0.000000e+00,NaN,roi_melodic_ic,g1,lifg,True
melodic_g1_cmap_lstg_oic_1,0.046938,0.008290,0.951049,0.010222,-22404.857,6.446500e-10,NaN,roi_melodic_ic,g1,lstg,True
melodic_g2_cmap_lstg_oic_1,0.045639,0.008562,0.952520,0.010510,-21715.949,5.493700e-09,NaN,roi_melodic_ic,g2,lstg,True
melodic_g2_cmap_lifg_oic_1,0.045289,0.008397,0.952780,0.010317,-22423.876,1.072000e-08,NaN,roi_melodic_ic,g2,lifg,True
melodic_g1_cmap_lstg_oic_0,0.043451,0.008220,0.955655,0.010204,-22432.545,6.761900e-09,NaN,roi_melodic_ic,g1,lstg,True
melodic_g1_cmap_lstg_oic_2,0.039185,0.008156,0.959145,0.010189,-22417.855,1.953600e-07,NaN,roi_melodic_ic,g1,lstg,True
melodic_g2_cmap_lstg_oic_2,0.032639,0.008247,0.966077,0.010385,-21735.657,1.326300e-05,NaN,roi_melodic_ic,g2,lstg,True
melodic_g2_cmap_lifg_oic_0,0.025841,0.007993,0.971968,0.010177,-22428.968,3.793500e-04,NaN,roi_melodic_ic,g2,lifg,True
melodic_g1_cmap_lifg_oic_2,0.022726,0.007869,0.976472,0.010093,-22701.667,1.531200e-03,NaN,roi_melodic_ic,g1,lifg,True
melodic_g2_cmap_lstg_oic_0,0.017437,0.008127,0.980568,0.010424,-21725.825,1.466100e-02,NaN,roi_melodic_ic,g2,lstg,True


In [11]:
import numpy as np
def calculate_meta(h2s, standard_errors):
    """
    Calculate meta-analytic h2-statistic using inverse-variance weighting
    
    Parameters:
    values: array of heritability estimates
    standard_errors: array of corresponding standard errors
    
    Returns:
    meta_h2: weighted average z-statistic
    meta_se: standard error of the meta-analytic estimate
    """
    
    # Calculate weights (inverse variance)
    weights = 1 / (standard_errors ** 2)
    
    # Meta-analytic weighted average z-statistic
    meta_h2 = np.sum(weights * h2s) / np.sum(weights)
    
    # Standard error of meta-analytic estimate
    meta_se = 1 / np.sqrt(np.sum(weights))
    
    return meta_h2, meta_se

In [12]:
print(calculate_meta(h2_data["Vg"].values, h2_data["Vg_SE"].values))

(np.float64(0.03386851096986693), np.float64(0.0008317219054159685))


In [14]:
h2_data[h2_data["idp_type"] == "roi_melodic_ic"].to_csv(os.path.join(workspace_path, "CONGRADS_rest", "results", "heritability_df_export.csv"))